## download_sources
Bronze-layer data acquisition: downloads raw files from FRED / Zillow / FHFA / Realtor into their Unity Catalog Volumes (`{catalog}.raw.<source>`). Thin composition root — all logic lives in the `data_fetch` package under `libs/`.

Batch policy is **abort-on-first**: `run_all` raises on the first file that fails, so the task fails fast and the orchestrator sees it. Per-file audit is the `{catalog}.audit.download_log` table; this notebook also writes one `pipeline_step_log` row via `StepLog` so a download failure reaches `pipeline_log_finalize` (design §6).

Pull ALL available history — no date windowing.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# Open the pipeline_step_log row for this download step (design §6). notebook_init
# injected Utils, StepLog, AUDIT, PIPELINE_RUN_ID. The step_log_id is reused as the FK
# for every download_log row so the file-level audit chains to this step.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = 1,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = None,   # downloads land files in Volumes; per-file detail is in download_log
)

In [ ]:
# Composition root (design §8.1). notebook_init injected CATALOG, AUDIT, RAW_FILES,
# STATUS_*, PIPELINE_RUN_ID, spark, dbutils, datetime, uuid, StepLog. The data_fetch
# package is environment-agnostic; this cell wires the Databricks-specific collaborators.
import tempfile
from functools import partial

from data_fetch import (
    run_all, SOURCES, RunContext, VolumeFileWriter, DownloadJournal,
    DatabricksSecretResolver,
)
from pipeline_logging import download_log_insert, download_log_last_sha256

try:
    ctx = RunContext(
        catalog=CATALOG,
        pipeline_run_id=str(PIPELINE_RUN_ID),
        step_log_id=step.step_log_id,        # FK to the pipeline_step_log row opened above
        audit_schema=AUDIT,
        scratch_dir=tempfile.gettempdir(),   # serverless-safe; NEVER /local_disk0 (§16.8)
        now=lambda: datetime.now(timezone.utc),
    )

    # abort-on-first: run_all raises on the first failed file (caught below → step.fail).
    summary = run_all(
        SOURCES, ctx,
        writer=VolumeFileWriter(RAW_FILES),                      # base path from notebook_init
        journal=DownloadJournal(
            record=partial(download_log_insert, spark, AUDIT),
            last_sha256=partial(download_log_last_sha256, spark, AUDIT),
        ),
        secrets=DatabricksSecretResolver(dbutils, scope="marketpulse"),  # reads FRED key via dbutils.secrets.get(scope, key)
    )
    print(summary.describe())

    step.rows_read    = len(summary.outcomes)                    # files attempted
    step.rows_written = len(summary.by_status(STATUS_SUCCEEDED)) # files newly landed
    step.succeed()

except Exception as e:
    step.fail(e); raise